# A first look at `pysignet`


This notebook shows the basic concepts of the `pysignet` library with a simple logical constraint and neural predicates.


A core concept in `pysignet` is the idea of a [named neuron](https://svivek.com/writing/2026-01-28-named-neurons.html). *A named neuron is a scalar activation in a neural network that admits a mapping to a symbol in a formal system, enabling logical reasoning over the network.*

In other words, these are specific neurons that we give symbolic names (like `P`, `Q`, `Digit`, etc) and use in logical expressions. These can be 
- Neural network outputs (i.e. final layer activations)
- Determninistic of inputs
- Intermediate activations


Importantly, named neurons represent predicates whose arguments are the inputs to the neural networks. So if an input is `X`, we can write `P(X)`, which would be mapped to a neuron.

This example uses only output named neurons. That is, it only uses final outputs of neural netowrks and functions that we can combine with logical operators. Other notebooks show more complex behaviors.

First, let us get some imports out of the way. The library is built on PyTorch, so we have the `torch` imports. We import `pysignet as psn`, which gives us everything we need: `Symbol`, `Variable`, and the logical operators `And`, `Or`, `Not`, `Implies`, `Equivalent`, `ForAll`, and `Exists`.

In [ ]:
import torch
import torch.nn as nn 
import pysignet as psn

## The logical expression

To get started, let us create the predictes and variables to construct the following logical expression:
$$\forall X, P(X) \wedge \left(Q(X) \vee \neg R(X)\right)$$

Here, we have three predicates `P`, `Q` and `R`, all of which operate on the same argument `X`. The rule says that for any input `X`, the predicate `P` should hold and either the predicate `Q` should hold or the predicate `R` should *not*. 

First, let us create the predicates and the variable.

In [2]:
P, Q, R = psn.Symbol ("P Q R")
X = psn.Variable("X")


We can now use `pysignet`'s boolean operators `And`, `Or` and `Not` (re-exported from SymPy for convenience) to construct the expression above.

Now we encounter the first design choice in `pysignet`: Any variable that is not explicitly bound to a quantifier is assumed to be universally quantified. So we need not write the universal quantification explicitly when we construct the logical expression.

> **Power users**: `pysignet` re-exports the standard SymPy boolean operators `And`, `Or`, `Not`, `Implies`, and `Equivalent` so you don't need to `import sympy as sp` for basic use. If you need advanced SymPy features (simplification, CNF conversion, etc.), you can still `import sympy as sp` — the operators are the same objects.

In [ ]:
expr = psn.And(P(X), psn.Or(Q(X), psn.Not(R(X))))

print(f"Logic expression: {expr}")

## The neural networks

At this point, all we have is a purely symbolic expression. Next, let us create the "neural" parts of this neuro-symbolic synthesis. 

First, let us create a neural network that computes a soft version of the predicate `P`.  For this exercise, it is a simple two layer MLP.

In [4]:
model_p = nn.Sequential(nn.Linear(10, 5), 
                nn.ReLU(), 
                nn.Linear(5, 1),
                nn.Sigmoid())

The predicates `Q` and `R` can also be mapped to neural networks. But the library allows for more: they can be mapped to *any* computation graphs we construct using torch with or without learnable parameters. 

To illustrate that, we can create two deterministic functions of the inputs:

* The deterministic function `pred_q` whose ouput becomes the predicate `Q` for any given `x`
* Another deterministic function `pred_r` whose output becomes the predicate `R` for any given `x`

In [7]:
def pred_q(x: torch.Tensor) -> torch.Tensor:
    return (x.sum(dim=-1, keepdim=True) > 0).float()

def pred_r(x: torch.Tensor) -> torch.Tensor:
    return torch.sigmoid(x.mean(dim=-1, keepdim=True)).squeeze(-1)

## Binding the neural and the symbolic expressions

Now, we are ready to instantiate the named neuron abstraction. To do that, we need to map the symbolic predicates to their soft counterparts with a python `dict`. The mapping admits any functions, lambdas or `nn.Module`s. We have mapped the predicate `Q` to a lambda that takes one argument because the predicate takes one argument `X`.

In [9]:
predicates = {
    "P": model_p,
    "Q": lambda x: pred_q(x).squeeze(-1),
    "R": pred_r
}

## From logic to differentiable logic

At this point, we have all the pieces we need to convert the logic into differentiable units. `pysignet` provides a single convenience function `compile_logic` to compile logic to differentiable units. 

In [10]:
compiled = psn.compile_logic(expr, predicates)

What we have done is to construct a computation graph that is a soft version of the boolean expression, where the truth values of the predicates are computed by their named neurons. By default, the computation graph uses the R-product t-norm relaxation, but `compile_logic` allows other compilers.

We can use the compiled expression to check soft-satisfaction of the predicates for any data. For this example, let us use a random batch of data. Let us now create the data.

In [11]:
batch_size = 32
x = torch.randn(batch_size, 10)

We still need to tell the expression that the variable `X` in the logical expression maps to the `x` that we have constructed. The compiled logic provides a natural way to do this. First, we can ask for the boolean value of the logical expression, given the decisions about the predicates `P`, `Q` and `R`.

In [12]:
satisfaction = compiled(X=x, return_boolean=True)
satisfaction

tensor([False, False, False, False, False, False, False, False, False, False,
         True,  True, False,  True, False,  True, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False])

But the more interesting thing happens when we remove the `return_boolean` flag to get a soft satisfaction. This gives us the result of a forward pass of the computation graph corresponding to our logical expression, with the variable `X` is bound to `x`, and the predicates bound to their respective named neurons.

In [13]:
soft_satisfaction = compiled(X=x)
soft_satisfaction

tensor([0.3375, 0.1955, 0.4477, 0.3823, 0.4044, 0.1999, 0.2615, 0.3771, 0.2064,
        0.3379, 0.5088, 0.3580, 0.3157, 0.5098, 0.2127, 0.2730, 0.3891, 0.2347,
        0.1953, 0.2266, 0.3718, 0.2460, 0.1950, 0.4683, 0.4837, 0.4578, 0.2283,
        0.3710, 0.3528, 0.4151, 0.3303, 0.3779], grad_fn=<ProdBackward1>)

### Technical note: Why soft and boolean satisfaction differ

You may notice that many `soft_satisfaction` values are below 0.5, yet the corresponding `satisfaction` (boolean) values are `True`. This is expected behavior! It arises due to the fundamental difference between the t-norm evaluation that populates the soft values, and boolean logic.

Consider `P(X) & Q(X)` where P=0.7 and Q=0.6:

| Method | Calculation | Result |
|--------|-------------|--------|
| **Soft (product t-norm)** | $0.7 \times 0.6$ | **0.42** |
| **Boolean** | (0.7 > 0.5) AND (0.6 > 0.5) | **True** |

The product t-norm *multiplies* satisfaction values, so even when all predicates are above 0.5, the conjunction can drop below 0.5. With three predicates each at 0.6: $0.6 \times 0.6 \times 0.6 = 0.216$.

For our expression `P(X) & (Q(X) | ~R(X))`, if P=0.6, Q=0.55, R=0.45, we have:
- **Soft**: $0.6 \times (0.55 + 0.55 - 0.55\times0.55) = 0.6 \times 0.80 = 0.48$
- **Boolean**: `True AND (True OR True) = True`

The two kinds of satisfaction ask different questions, and as a result we get different answers: 

Soft satisfaction measures the *degree* of truth, which is useful for computing gradients during training. The question it answers is "How well is the constraint satisfied?"

Boolean satisfaction** measures *hard* decisions, useful for evaluation metrics. The question it answers is "Is the constraint satisfied according to model predictions?"

This may be tricky, but the good news, however, is that when soft satisfaction increases, so will boolean satisfaction. This means that a natural way to train neural networks is to try to define a loss such that minimizing it maximizes soft satisfaction. 

## From logic to loss

We could use the logical expressions to automaticaly construct differentiable loss functions. We have a helper `logic_to_loss` to help with that. The interface of `logic_to_loss` is similar to that of `compile_logic` in that it takes the logical expression and the mapping from predicates to neurons, and generates a compiled expression that can be applied to a batch of examples.

In [14]:
loss_calculator = psn.logic_to_loss(expr, predicates)

Given a batch of examples `x` (we will re-use the batch we created above), we can now compute the value of loss. The loss can be interpreted as "how poorly does the current batch of examples satisfy the logical constraint". This means that minimizing the loss will lead to better satisfaction.

In [15]:
loss = loss_calculator.loss(X=x)
print(f"Loss = {loss}")

Loss = 36.629241943359375


The loss and the compiled expression underneath the hood are all differentiable. This means that we can use the standard torch machinery like doing a backward pass, and using any optimizer we want to improve the objective.

In [16]:
loss.backward()
print(f"\nGradients computed: {model_p[0].weight.grad is not None}")


Gradients computed: True


We will see how to use this to train neural networks that satisfy properties in other notebooks. A good next step is to see [the notebook that shows how we can build a standard MNIST classifier with this library](MNIST.ipynb). 